In [1]:
# === Core Python ===
import os
import glob
import re
import collections
import cftime
from datetime import datetime
from typing import Tuple, Dict, Optional, List

# === Numerical & Data Handling ===
import numpy as np
import numpy.ma as ma
import pandas as pd
import xarray as xr
import xcdat as xcd
import xskillscore as xs
from xskillscore import rmse, pearson_r

from dataclasses import dataclass

from DataUtil import (
    build_experiments,
    DEFAULT_EXPERIMENTS
)

from ObsUtil import (
    OBS_REGISTRY,
    get_obs_file,
    list_obs_sources,
    obs_coverage,
)

from time_window import(
    build_windows_from_start
)


ERROR! Session/line number was not unique in database. History logging moved to new session 611


In [2]:
@dataclass(frozen=True)
class FileCollectionConfig:
    group: str
    freq: str
    run: str
    obs: str
    period: str
    model_template: str            # <-- REQUIRED, explicit
    ens_prefix: str = "en"
    ens_width: int = 2
    ens_start: int = 1

    def parse_period(self) -> Tuple[int, int, int, int]:
        m = re.match(r"^(\d{4})(\d{2})-(\d{4})(\d{2})$", self.period)
        if not m:
            raise ValueError(f"Bad period '{self.period}', expected 'YYYYMM-YYYYMM'")
        y0, m0, y1, m1 = map(int, m.groups())
        if (y1, m1) < (y0, m0):
            raise ValueError(f"Bad period '{self.period}' (end < start)")
        return y0, m0, y1, m1

    def years(self) -> List[int]:
        y0, _, y1, _ = self.parse_period()
        return list(range(y0, y1 + 1))

    def ens_labels(self, nens: int) -> List[str]:
        # FIX: use self.ens_start (was ens_start undefined)
        return [
            f"{self.ens_prefix}{i:0{self.ens_width}d}"
            for i in range(self.ens_start, nens + self.ens_start)
        ]


class S2SFileCollector:
    """
    Collect obs + model files for S2S / ACC diagnostics.
    """

    def __init__(
        self,
        *,
        exp_list: dict,
        exp_dict: dict,
        obs_registry,
        s2s_var_dict: Dict[str, str],
        get_obs_file_func,
    ):
        self.exp_list = exp_list
        self.exp_dict = exp_dict
        self.obs_registry = obs_registry
        self.s2s_var_dict = s2s_var_dict
        self.get_obs_file = get_obs_file_func

    # ---------- path resolvers ----------

    def resolve_obs_file(
        self,
        obs: str,
        freq: str,
        year: int,
        var: Optional[str] = None,
    ) -> str:
        return self.get_obs_file(
            self.obs_registry,
            obs,
            freq=freq,
            year=year,
            var=var,
        )

    def model_ts_dir(self, run_meta, freq: str) -> str:
        return os.path.join(run_meta.atm_path, "ts", freq)

    def resolve_model_file(
        self,
        *,
        run_meta,
        freq: str,
        year: int,
        var: str,
        ens: str,
        template: str,
    ) -> str:
        ts_dir = self.model_ts_dir(run_meta, freq)
        fname = template % {
            "var": var,
            "ens": ens,
            "year": year,
            "period": run_meta.period,
        }
        return os.path.join(ts_dir, fname)

    # ---------- model file discovery ----------

    def candidates_model_files(
        self,
        *,
        ts_dir: str,
        var: str,
        ens: str,
        years: List[int],
        period: str,
        templates: List[str],
    ) -> List[str]:
        out: List[str] = []
        for tpl in templates:
            for y in years:
                name = tpl % {
                    "var": var,
                    "ens": ens,
                    "year": y,
                    "period": period,
                }
                path = os.path.join(ts_dir, name)
                if os.path.exists(path):
                    out.append(path)

        # de-duplicate while preserving order
        seen = set()
        uniq: List[str] = []
        for p in out:
            if p not in seen:
                uniq.append(p)
                seen.add(p)
        return uniq

    def model_files_for_ens(
        self,
        cfg: FileCollectionConfig,
        *,
        run_meta,
        var: str,
        ens: str,
    ) -> List[str]:
        ts_dir = self.model_ts_dir(run_meta, cfg.freq)
        years = cfg.years()
        return [
            os.path.join(
                ts_dir,
                cfg.model_template % {"var": var, "ens": ens, "year": y, "period": cfg.period},
            )
            for y in years
        ]

    # ---------- main collection ----------

    def collect_one_var(
        self,
        cfg: FileCollectionConfig,
        *,
        var: str,
        obs_var: str,
        verbose: bool = True,
    ):
        years = cfg.years()
        models = self.exp_list[cfg.group]["models"]

        # --- obs files ---
        obs_files = [self.resolve_obs_file(cfg.obs, cfg.freq, y, obs_var) for y in years]
        missing_obs = [f for f in obs_files if not os.path.exists(f)]
        if missing_obs:
            if verbose:
                print(f"[MISSING OBS] {var} (obs_var={obs_var})")
                for f in missing_obs:
                    print("  ", f)
            return None

        out_var = {}

        for exp in models:
            run_meta = self.exp_dict[exp]["runs"][cfg.run]
            if run_meta is None:
                if verbose:
                    print(f"SKIP {exp}: no run='{cfg.run}'")
                continue

            nens = int(self.exp_dict[exp]["nens"])
            ts_dir = self.model_ts_dir(run_meta, cfg.freq)

            model_by_ens = {}
            missing_any = False

            for ens in cfg.ens_labels(nens):
                files = self.model_files_for_ens(cfg, run_meta=run_meta, var=var, ens=ens)

                missing = [f for f in files if not os.path.exists(f)]
                if missing:
                    missing_any = True
                    if verbose:
                        print(f"[MISSING MOD] {exp} {var} {ens} (ts_dir={ts_dir})")
                        for f in missing[:5]:
                            print("   ", f)
                        if len(missing) > 5:
                            print(f"   ... {len(missing)-5} more")

                model_by_ens[ens] = files  # keep full list for later loading

            out_var[exp] = {
                "obs": obs_files,
                "model": model_by_ens,
                "ts_dir": ts_dir,
                "nens": nens,
            }

            if verbose and missing_any:
                print(f"[WARN] {exp} {var}: some ensemble members missing files")

        return out_var

    def collect(
        self,
        cfg: FileCollectionConfig,
        *,
        vars_to_process: Optional[List[str]] = None,
        verbose: bool = True,
    ) -> Dict[str, dict]:
        out: Dict[str, dict] = {}
        for var, obs_var in self.s2s_var_dict.items():
            if vars_to_process is not None and var not in vars_to_process:
                continue

            res = self.collect_one_var(cfg, var=var, obs_var=obs_var, verbose=verbose)
            if res is None:
                continue
            out[var] = res

        return out

FLUX_SIGN_REGISTRY = {
    ("ERA5",  "FLNS"):  +1,
    ("ERA5",  "FLNSC"): +1,
    ("ERA5",  "FSNS"):  +1,
    ("ERA5",  "FSNSC"): +1,
    ("ERA5",  "FLDS"):  +1,
    ("ERA5",  "FSDS"):  +1,
    ("ERA5",  "FLUT"):  +1,

    ("MODEL", "FLNS"):  +1,
    ("MODEL", "FLNSC"): +1,
    ("MODEL", "FSNS"):  +1,
    ("MODEL", "FSNSC"): +1,
    ("MODEL", "FLDS"):  +1,
    ("MODEL", "FSDS"):  +1,
    ("MODEL", "FLUT"):  +1,
}

@dataclass(frozen=True)
class WindowedTCCBaseConfig:
    windows: Dict[str, Tuple[str, str]]  # label -> (start_date, end_date), ISO yyyy-mm-dd
    regions: Optional[List[dict]] = None  # [{"name":..., "lat":(lo,hi), "lon":(lo,hi)}]
    lat_range: Optional[Tuple[float, float]] = None
    lon_range: Optional[Tuple[float, float]] = None

    # grid/time checks
    require_same_grid: bool = True
    grid_tol: float = 0.0
    require_nonempty_time: bool = True

    # output
    out_dir: str = "./base_tcc_rmse"
    overwrite: bool = False
    verbose: bool = True
    
@dataclass(frozen=True)
class WindowedTCCAggregateConfig:
    member_quantiles: Tuple[float, ...] = (0.1, 0.5, 0.9)
    overwrite: bool = True
    verbose: bool = True

class S2SWindowedTCCBaseBuilder:
    """
    Stage 1: build base files per (exp, var, ens).
      output file contains:
        - TCC(window, lat, lon)
        - RMSE(window, lat, lon)
        - coords: window_start/window_end
        - attrs: exp/ens/var/region/period
    """

    def __init__(self, cfg: WindowedTCCBaseConfig):
        self.cfg = cfg
        os.makedirs(self.cfg.out_dir, exist_ok=True)
        
    # ---------- helpers ----------

    @staticmethod
    def _parse_yyyymm_period(period: str) -> Tuple[int, int, int, int]:
        m = re.match(r"^(\d{4})(\d{2})-(\d{4})(\d{2})$", period)
        if not m:
            raise ValueError(f"Bad period '{period}', expected 'YYYYMM-YYYYMM'")
        y0, m0, y1, m1 = map(int, m.groups())
        return y0, m0, y1, m1

    @classmethod
    def _select_period_time(cls, da: xr.DataArray, period: str) -> xr.DataArray:
        y0, m0, y1, m1 = cls._parse_yyyymm_period(period)
        t0 = np.datetime64(f"{y0:04d}-{m0:02d}-01")
        if m1 == 12:
            t1 = np.datetime64(f"{y1 + 1:04d}-01-01")
        else:
            t1 = np.datetime64(f"{y1:04d}-{m1 + 1:02d}-01")
        return da.sel(time=slice(t0, t1))

    @staticmethod
    def _to_daily_date_time(da: xr.DataArray) -> xr.DataArray:
        if "time" not in da.coords:
            raise KeyError("Missing time coordinate.")
        t = da["time"]
    
        if np.issubdtype(t.dtype, np.datetime64):
            t_daily = t.astype("datetime64[D]")
        else:
            t_daily = xr.DataArray(
                np.array([np.datetime64(str(v)[:10]) for v in t.values], dtype="datetime64[D]"),
                dims=t.dims, coords=t.coords, name=t.name
            )
    
        out = da.assign_coords(time=t_daily)
    
        # NEW: collapse duplicates after daily casting
        if out.indexes["time"].has_duplicates:
            out = out.groupby("time").mean("time")
    
        return out

    @staticmethod
    def _open_var(files: List[str], var: str) -> xr.DataArray:
        ds = xr.open_mfdataset(files, combine="by_coords")
        if var in ds:
            return ds[var]
        for k in ds.data_vars:
            if k.lower() == var.lower():
                return ds[k]
        raise KeyError(f"Variable '{var}' not found in files. Available={list(ds.data_vars)}")

    @staticmethod
    def _guess_lat_lon_names(da: xr.DataArray) -> Tuple[str, str]:
        lat_candidates = ["lat", "latitude", "LAT", "nlat"]
        lon_candidates = ["lon", "longitude", "LON", "nlon"]
        lat = next((n for n in lat_candidates if (n in da.dims or n in da.coords)), None)
        lon = next((n for n in lon_candidates if (n in da.dims or n in da.coords)), None)
        if lat is None or lon is None:
            raise ValueError(f"Cannot infer lat/lon. dims={da.dims} coords={list(da.coords)}")
        return lat, lon

    @staticmethod
    def _wrap_lon_0_360(da: xr.DataArray, lon_name: str) -> xr.DataArray:
        if lon_name not in da.coords:
            return da
        lon = da[lon_name]
        if not np.issubdtype(lon.dtype, np.number):
            return da
        lon2 = (lon % 360.0)
        return da.assign_coords({lon_name: lon2}).sortby(lon_name)

    @staticmethod
    def _subset_region(da: xr.DataArray, lat: str, lon: str,
                       lat_range: Optional[Tuple[float, float]],
                       lon_range: Optional[Tuple[float, float]]) -> xr.DataArray:
        out = da
        if lat_range is not None:
            lo, hi = lat_range
            latv = out[lat]
            if latv.size > 1 and (latv[0] > latv[-1]):
                out = out.sel({lat: slice(hi, lo)})
            else:
                out = out.sel({lat: slice(lo, hi)})
        if lon_range is not None:
            lo, hi = lon_range
            out = out.sel({lon: slice(lo, hi)})
        return out

    @staticmethod
    def _same_1d_values(a: xr.DataArray, b: xr.DataArray, tol: float) -> bool:
        if a.size != b.size:
            return False
        av = np.asarray(a.values)
        bv = np.asarray(b.values)
        if tol == 0.0:
            return np.array_equal(av, bv)
        return np.allclose(av, bv, rtol=0.0, atol=tol, equal_nan=True)

    @staticmethod
    def _rename_dim_if_present(da: xr.DataArray, old: str, new: str) -> xr.DataArray:
        if old == new:
            return da
        if (new in da.dims) or (new in da.coords):
            return da
        rename_map = {}
        if old in da.dims:
            rename_map[old] = new
        if old in da.coords:
            rename_map[old] = new
        return da.rename(rename_map) if rename_map else da

    def _ensure_names_match(self, mod: xr.DataArray, obs: xr.DataArray,
                            mod_lat: str, mod_lon: str, obs_lat: str, obs_lon: str) -> xr.DataArray:
        if (mod_lat == obs_lat) and (mod_lon == obs_lon):
            return mod
        lat_ok = self._same_1d_values(mod[mod_lat], obs[obs_lat], self.cfg.grid_tol)
        lon_ok = self._same_1d_values(mod[mod_lon], obs[obs_lon], self.cfg.grid_tol)
        if self.cfg.require_same_grid and not (lat_ok and lon_ok):
            raise ValueError("Model/obs grids differ; regrid explicitly or set require_same_grid=False.")
        mod = self._rename_dim_if_present(mod, mod_lat, obs_lat)
        mod = self._rename_dim_if_present(mod, mod_lon, obs_lon)
        return mod

    def _align_and_check(self, mod: xr.DataArray, obs: xr.DataArray) -> tuple[xr.DataArray, xr.DataArray]:
        mod, obs = xr.align(mod, obs, join="inner")
        mod = mod.sortby("time")
        obs = obs.sortby("time")
        if self.cfg.require_nonempty_time and mod.sizes.get("time", 0) == 0:
            raise ValueError("No overlapping times between model and obs after alignment.")
        return mod, obs

    # ---------- kernels ----------
    @staticmethod
    def _maybe_convert_precip_to_mmday(da: xr.DataArray, *, var_key: str, verbose: bool=False) -> xr.DataArray:
        k = (var_key or "").lower()
        name = (da.name or "").lower()

        is_pr = (
            k in ("pr","precip","prect","precc","precl","precsl","tp")
            or "precip" in k or "precip" in name
            or k.startswith("pr") or name.startswith("pr")
            or name in ("prect","precc","precl","precsl","tp")
        )
        if not is_pr:
            return da

        u = (da.attrs.get("units") or "").strip().lower()
        u0 = u.replace(" ", "")

        def _set(out, note):
            out.attrs = dict(da.attrs)
            out.attrs["units"] = "mm/day"
            out.attrs["note_unit_fix"] = note
            return out

        # already mm/day-ish
        if u0 in ("mm/day","mm/d","mmd-1","mmdy-1") or ("mm" in u0 and ("day" in u0 or "d-1" in u0)):
            return da

        # kg m-2 s-1 -> mm/day (1 kg/m2 = 1 mm water)
        if ("kg" in u0 and "m-2" in u0 and "s-1" in u0) or u0 in ("kgm-2s-1","kg/m2/s","kgm**-2s**-1"):
            if verbose:
                print(f"[UNIT FIX] {da.name or var_key}: kg m-2 s-1 -> mm/day (×86400)")
            return _set(da * 86400.0, "converted from kg m-2 s-1 via ×86400")

        # m/s -> mm/day
        if u0 in ("m/s","ms-1") or ("m" in u0 and "s-1" in u0 and "kg" not in u0 and "mm" not in u0):
            if verbose:
                print(f"[UNIT FIX] {da.name or var_key}: m/s -> mm/day (×86400×1000)")
            return _set(da * 86400.0 * 1000.0, "converted from m s-1 via ×86400×1000")

        # mm/s -> mm/day
        if u0 in ("mm/s","mms-1"):
            if verbose:
                print(f"[UNIT FIX] {da.name or var_key}: mm/s -> mm/day (×86400)")
            return _set(da * 86400.0, "converted from mm s-1 via ×86400")

        # m/day -> mm/day
        if u0 in ("m/day","m/d","md-1"):
            if verbose:
                print(f"[UNIT FIX] {da.name or var_key}: m/day -> mm/day (×1000)")
            return _set(da * 1000.0, "converted from m day-1 via ×1000")

        # if units missing: leave as-is (or add a heuristic if you want)
        return da
        
    @staticmethod
    def _maybe_convert_geopotential_to_height(
        da: xr.DataArray,
        *,
        var_key: str,
        g: float = 9.80665,
        verbose: bool = False,
    ) -> xr.DataArray:
        k = (var_key or "").lower()
        name = (da.name or "").lower()

        looks_like_z = (
            k.startswith("z")
            or "geopot" in k
            or "geopot" in name
            or name.startswith("z")
            or name in ("zg", "phi")
        )
        if not looks_like_z:
            return da

        units = (da.attrs.get("units") or "").strip().lower()
        u0 = units.replace(" ", "").replace("^", "").replace("**", "")

        def is_geopot(u: str) -> bool:
            return ("m2s-2" in u) or ("m2/s2" in u)

        def is_height(u: str) -> bool:
            return u in ("m", "meter", "metre", "meters", "metres")

        if u0:
            if is_geopot(u0):
                if verbose:
                    print(f"[UNIT FIX] {da.name or var_key}: geopotential -> height (÷g)")
                out = da / g
                out.attrs = dict(da.attrs)
                out.attrs["units"] = "m"
                out.attrs["note_unit_fix"] = "converted from geopotential (m2 s-2) via /g"
                return out
            if is_height(u0):
                return da

        # heuristic if units missing/unknown
        try:
            vmax = float(da.max(skipna=True))
        except Exception:
            return da

        if 2.0e4 < vmax < 5.0e6:
            if verbose:
                print(f"[UNIT FIX] {da.name or var_key}: heuristic geopotential -> height (÷g), vmax={vmax:.3g}")
            out = da / g
            out.attrs = dict(da.attrs)
            out.attrs["units"] = "m"
            out.attrs["note_unit_fix"] = "heuristic geopotential->height via /g (units missing/unknown)"
            return out

        return da

    def _maybe_fix_flux_sign_by_registry(
        self,
        da: xr.DataArray,
        *,
        var_key: str,
        source: str,   # "ERA5" / "MODEL" / "NOAA-OLR" ...
        registry: dict,
    ) -> xr.DataArray:
        v = (var_key or da.name or "").upper()

        s = registry.get((source, v), registry.get(v, +1))
        s = int(s)

        if s == -1:
            out = -da
            out.attrs = dict(da.attrs)
            out.attrs["note_sign_fix"] = f"flipped sign by registry: ({source},{v})"
            return out
        return da

    def _normalize_physics(self, da: xr.DataArray, *, var_key: str, source: str) -> xr.DataArray:
        """
        Normalize per-field physics once (no window loop cost):
          - lon to [0,360)
          - daily time coordinate
          - precip to mm/day (if needed)
          - geopotential -> height (if needed)
          - flux sign by registry (if needed)
        """
        lat, lon = self._guess_lat_lon_names(da)
        da = self._wrap_lon_0_360(da, lon)

        da = self._to_daily_date_time(da)

        da = self._maybe_convert_precip_to_mmday(da, var_key=var_key, verbose=self.cfg.verbose)
        da = self._maybe_convert_geopotential_to_height(da, var_key=var_key, verbose=self.cfg.verbose)

        da = self._maybe_fix_flux_sign_by_registry(
            da, var_key=var_key, source=source, registry=FLUX_SIGN_REGISTRY
        )
        return da

    @staticmethod
    def _tcc_over_time(f: xr.DataArray, o: xr.DataArray) -> xr.DataArray:
        fa = f - f.mean("time")
        oa = o - o.mean("time")
        num = (fa * oa).sum("time")
        den = xr.ufuncs.sqrt((fa * fa).sum("time") * (oa * oa).sum("time"))
        out = num / den
        return out.where(den > 0)

    @staticmethod
    def _rmse_over_time(f: xr.DataArray, o: xr.DataArray) -> xr.DataArray:
        return xr.ufuncs.sqrt(((f - o) ** 2).mean("time"))

    # ---------- filenames ----------

    def base_path(self, *, group: str, freq: str, run: str, obs: str, period: str,
                  region: str, exp: str, var: str, ens: str) -> str:
        fname = f"s2s_base_tcc_rmse_{group}_{freq}_{run}_{obs}_{period}_{region}_{exp}_{ens}_{var}.nc"
        return os.path.join(self.cfg.out_dir, fname)
        
    def ensmean_path(self, *, group: str, freq: str, run: str, obs: str, period: str,
                     region: str, exp: str, var: str) -> str:
        fname = f"s2s_base_tcc_rmse_{group}_{freq}_{run}_{obs}_{period}_{region}_{exp}_ENSMEAN_{var}.nc"
        return os.path.join(self.cfg.out_dir, fname)

    # ---------- main: build per member ----------

    def build_member_file(
        self,
        all_files: Dict[str, dict],
        *,
        group: str,
        freq: str,
        run: str,
        obs_name: str,
        period: str,
        var: str,
        exp: str,
        ens: str,
        obs_var_override: Optional[Dict[str, str]] = None,
    ) -> str:
        """
        Build ONE base file for a single (exp, var, ens) across all configured regions.
        """
        if var not in all_files or exp not in all_files[var]:
            raise KeyError(f"Missing var/exp in all_files: var={var} exp={exp}")

        # regions
        regions = self.cfg.regions or [dict(name="DEFAULT", lat=self.cfg.lat_range, lon=self.cfg.lon_range)]

        # open obs once
        any_exp = exp
        obs_files = all_files[var][any_exp]["obs"]
        obs_var = (obs_var_override or {}).get(var, var)
        obs0 = self._open_var(obs_files, obs_var)
        obs0 = self._select_period_time(obs0, period)
        obs0 = self._normalize_physics(obs0, var_key=var, source=obs_name)
        lat_obs, lon_obs = self._guess_lat_lon_names(obs0)
        
        # open member once
        model_files = all_files[var][exp]["model"].get(ens, [])
        if not model_files:
            raise FileNotFoundError(f"No model files for exp={exp} var={var} ens={ens}")

        mem = self._open_var(model_files, var)
        mem = self._select_period_time(mem, period)
        mem = self._normalize_physics(mem, var_key=var, source="MODEL")
        lat_m, lon_m = self._guess_lat_lon_names(mem)
        
        # per-region compute and pack
        reg_ds_list = []
        for reg in regions:
            rname = str(reg.get("name", "REGION"))
            rlat = reg.get("lat", None)
            rlon = reg.get("lon", None)

            obs_reg = self._subset_region(obs0, lat_obs, lon_obs, rlat, rlon)
            mem_reg = self._subset_region(mem, lat_m, lon_m, rlat, rlon)
            mem_reg = self._ensure_names_match(mem_reg, obs_reg, lat_m, lon_m, lat_obs, lon_obs)
            mem_a, obs_a = self._align_and_check(mem_reg, obs_reg)

            # windows
            wlabels = []
            wstart = []
            wend = []
            tcc_list = []
            rmse_list = []
            for wlab in sorted(self.cfg.windows.keys()):
                s, e = self.cfg.windows[wlab]
                ts = slice(np.datetime64(s), np.datetime64(e))
                f = mem_a.sel(time=ts)
                o = obs_a.sel(time=ts)
                if f.sizes.get("time", 0) == 0:
                    continue
                tcc_list.append(self._tcc_over_time(f, o))
                rmse_list.append(self._rmse_over_time(f, o))
                wlabels.append(wlab)
                wstart.append(np.datetime64(s))
                wend.append(np.datetime64(e))

            if not tcc_list:
                if self.cfg.verbose:
                    print(f"[SKIP] no windows produced for {exp} {ens} {var} region={rname}")
                continue

            TCC = xr.concat(tcc_list, dim=xr.IndexVariable("window", wlabels))
            RMSE = xr.concat(rmse_list, dim=xr.IndexVariable("window", wlabels))
            TCC = TCC.assign_coords(window_start=("window", np.array(wstart, dtype="datetime64[D]")),
                                    window_end=("window", np.array(wend, dtype="datetime64[D]")))
            RMSE = RMSE.assign_coords(window_start=TCC["window_start"], window_end=TCC["window_end"])

            ds_reg = xr.Dataset(
                data_vars=dict(TCC=TCC, RMSE=RMSE),
                coords=dict(window=TCC["window"]),
                attrs=dict(region=rname),
            ).assign_coords(region=rname).expand_dims("region")
            reg_ds_list.append(ds_reg)

        if not reg_ds_list:
            raise ValueError(f"No region results produced for exp={exp} ens={ens} var={var}")

        ds_out = xr.concat(reg_ds_list, dim="region")
        ds_out.attrs.update(
            dict(
                group=group,
                freq=freq,
                run=run,
                obs=obs_name,
                period=period,
                exp=exp,
                ens=ens,
                var=var,
                note="Base file: windowed TCC/RMSE maps for a single ensemble member.",
            )
        )

        # save
        path = self.base_path(
            group=group, freq=freq, run=run, obs=obs_name, period=period,
            region="ALL", exp=exp, var=var, ens=ens
        )
        if (not self.cfg.overwrite) and os.path.exists(path):
            if self.cfg.verbose:
                print(f"[SKIP EXISTS] {path}")
            return path

        ds_out.to_netcdf(path)
        if self.cfg.verbose:
            print(f"[SAVED] {path}")
        return path

    def build_ensmean_file(
        self,
        all_files: Dict[str, dict],
        *,
        group: str,
        freq: str,
        run: str,
        obs_name: str,
        period: str,
        var: str,
        exp: str,
        obs_var_override: Optional[Dict[str, str]] = None,
    ) -> str:
        """
        Build ONE base file for ensemble-mean-based windowed TCC/RMSE maps: (exp, var).
        """
        if var not in all_files or exp not in all_files[var]:
            raise KeyError(f"Missing var/exp in all_files: var={var} exp={exp}")

        regions = self.cfg.regions or [dict(name="DEFAULT", lat=self.cfg.lat_range, lon=self.cfg.lon_range)]

        # ---- open obs once ----
        obs_files = all_files[var][exp]["obs"]
        obs_var = (obs_var_override or {}).get(var, var)

        obs0 = self._open_var(obs_files, obs_var)
        obs0 = self._select_period_time(obs0, period)
        obs0 = self._normalize_physics(obs0, var_key=var, source=obs_name)
        lat_obs, lon_obs = self._guess_lat_lon_names(obs0)

        # ---- open + build ensemble mean once ----
        model_by_ens = all_files[var][exp]["model"]
        members = []
        ens_names = []
        for ens, files in model_by_ens.items():
            if not files:
                continue
            da = self._open_var(files, var)
            da = self._select_period_time(da, period)
            da = self._normalize_physics(da, var_key=var, source="MODEL")
            members.append(da)
            ens_names.append(ens)

        if not members:
            raise ValueError(f"No member data for exp={exp} var={var}")

        mod_ens = xr.concat(members, dim=xr.IndexVariable("ens", ens_names))
        mod_mean = mod_ens.mean("ens")   # already normalized member-wise
        lat_m, lon_m = self._guess_lat_lon_names(mod_mean)

        # ---- per-region compute ----
        reg_ds_list = []
        for reg in regions:
            rname = str(reg.get("name", "REGION"))
            rlat = reg.get("lat", None)
            rlon = reg.get("lon", None)

            obs_reg = self._subset_region(obs0, lat_obs, lon_obs, rlat, rlon)
            mod_reg = self._subset_region(mod_mean, lat_m, lon_m, rlat, rlon)

            mod_reg = self._ensure_names_match(mod_reg, obs_reg, lat_m, lon_m, lat_obs, lon_obs)
            mod_a, obs_a = self._align_and_check(mod_reg, obs_reg)
            
            wlabels, wstart, wend = [], [], []
            tcc_list, rmse_list = [], []
            for wlab in sorted(self.cfg.windows.keys()):
                s, e = self.cfg.windows[wlab]
                ts = slice(np.datetime64(s), np.datetime64(e))
                f = mod_a.sel(time=ts)
                o = obs_a.sel(time=ts)
                if f.sizes.get("time", 0) == 0:
                    continue
                tcc_list.append(self._tcc_over_time(f, o))
                rmse_list.append(self._rmse_over_time(f, o))
                wlabels.append(wlab)
                wstart.append(np.datetime64(s))
                wend.append(np.datetime64(e))

            if not tcc_list:
                if self.cfg.verbose:
                    print(f"[SKIP] ensmean: no windows produced for {exp} {var} region={rname}")
                continue

            TCC = xr.concat(tcc_list, dim=xr.IndexVariable("window", wlabels))
            RMSE = xr.concat(rmse_list, dim=xr.IndexVariable("window", wlabels))

            TCC = TCC.assign_coords(
                window_start=("window", np.array(wstart, dtype="datetime64[D]")),
                window_end=("window", np.array(wend, dtype="datetime64[D]")),
            )
            RMSE = RMSE.assign_coords(window_start=TCC["window_start"], window_end=TCC["window_end"])

            ds_reg = xr.Dataset(
                data_vars=dict(TCC_ensmean=TCC, RMSE_ensmean=RMSE),
                coords=dict(window=TCC["window"]),
                attrs=dict(region=rname),
            ).assign_coords(region=rname).expand_dims("region")

            reg_ds_list.append(ds_reg)

        if not reg_ds_list:
            raise ValueError(f"No ensmean region results produced for exp={exp} var={var}")

        ds_out = xr.concat(reg_ds_list, dim="region")
        ds_out.attrs.update(
            dict(
                group=group, freq=freq, run=run, obs=obs_name, period=period,
                exp=exp, var=var,
                note="Ensemble-mean-based base file: windowed TCC/RMSE maps computed from ensemble-mean time series.",
            )
        )

        path = self.ensmean_path(
            group=group, freq=freq, run=run, obs=obs_name, period=period,
            region="ALL", exp=exp, var=var
        )
        if (not self.cfg.overwrite) and os.path.exists(path):
            if self.cfg.verbose:
                print(f"[SKIP EXISTS] {path}")
            return path

        ds_out.to_netcdf(path)
        if self.cfg.verbose:
            print(f"[SAVED] {path}")
        return path
        
class S2SWindowedTCCAggregator:
    """
    Stage 2: aggregate member base files into ensemble summary for each (exp,var).
    Produces:
      - TCC_mean/std/quantile over ens
      - RMSE_mean/std/quantile over ens
    """

    def __init__(self, cfg: WindowedTCCAggregateConfig):
        self.cfg = cfg

    @staticmethod
    def _open_many(paths: List[str]) -> xr.Dataset:
        return xr.open_mfdataset(paths, combine="by_coords")

    def aggregate_exp_var(
        self,
        member_paths: List[str],
        *,
        out_nc: str,
        ensmean_path: str | None = None,
    ) -> str:

        ds_list = []
        ens_names = []
        for p in member_paths:
            ds = xr.open_dataset(p)
            ens = str(ds.attrs.get("ens", "ens"))
            ds_list.append(ds.load())
            ds.close()
            ens_names.append(ens)

        ds_ens = xr.concat(ds_list, dim=xr.IndexVariable("ens", ens_names))

        out = xr.Dataset()
        out["TCC_member"]  = ds_ens["TCC"]
        out["RMSE_member"] = ds_ens["RMSE"]

        out["TCC_mean"] = out["TCC_member"].mean("ens")
        out["RMSE_mean"] = out["RMSE_member"].mean("ens")
        out["TCC_std"]  = out["TCC_member"].std("ens")
        out["RMSE_std"] = out["RMSE_member"].std("ens")

        qs = list(self.cfg.member_quantiles or ())
        if qs:
            out["TCC_quantile"] = out["TCC_member"].quantile(qs, dim="ens").rename({"quantile": "q"})
            out["RMSE_quantile"] = out["RMSE_member"].quantile(qs, dim="ens").rename({"quantile": "q"})

        # add ensmean diagnostics if provided
        if ensmean_path is not None and os.path.exists(ensmean_path):
            ds_em = xr.open_dataset(ensmean_path)
            if "TCC_ensmean" in ds_em and "RMSE_ensmean" in ds_em:
                out["TCC_ensmean"] = ds_em["TCC_ensmean"]
                out["RMSE_ensmean"] = ds_em["RMSE_ensmean"]
            ds_em.close()

        # attrs
        for k, v in ds_list[0].attrs.items():
            out.attrs[k] = v
        out.attrs["note"] = "Ensemble aggregation from per-member base files (mean/std/quantiles) + optional ensmean-based diagnostics."

        if (not self.cfg.overwrite) and os.path.exists(out_nc):
            raise FileExistsError(f"Output exists: {out_nc}")

        out.to_netcdf(out_nc)
        if self.cfg.verbose:
            print(f"[SAVED] {out_nc}")
        return out_nc

In [3]:
if __name__ == "__main__":
    data_path = "/compyfs/zhan391/v3_dart_cda_scratch"
    out_path  = "/compyfs/zhan391/v3_dart_cda_scratch/diag_output/tcc_rmse_base"
    os.makedirs(out_path, exist_ok=True)
    exp_list = {
        "Jan2012": dict(
            models=["CTRL10-S0", "CAPT10-S0", "DART20-S0", "DART40-S0"],
            period="201201-201203",
            season="Winter",
            init_date="2012-01-01",
            init_month=1,
            run="fc",
            freq="daily",
        ),
        "Jun2012": dict(
            models=["CTRL10-S1", "CAPT10-S1", "DART40-S1"],
            period="201206-201208",
            season="Summer",
            init_date="2012-06-01",
            init_month=6,
            run="fc",
            freq="daily",
        ),
    }

    # Unique labels (your previous dict overwrote itself)
    lead_bins = {
        "week1-2": (1, 14),
        "week3-4": (15, 28),
        "week5-6": (29, 42),
        "week7-8": (43, 56),
    }
    
    # obs -> {model_var -> obs_var}
    s2s_var_dict = {
        "ERA5": {
            "UBOT": "U10",
            "VBOT": "V10",
            "PRECT": "PRECT",
            "PSL": "PSL",
            "U850": "U850",
            "V850": "V850",
            "T850": "T850",
            "Q850": "Q850",
            "Z500": "Z500",
            "OMEGA500": "OMEGA500",
            "OMEGA200": "OMEGA200",
            "OMEGA850": "OMEGA850",
            "U200": "U200",
            "V200": "V200",
            "U500": "U500",
            "V500": "V500",
            "FLDS": "FLDS",
            "FLDSC": "FLDSC",
            "FLNS": "FLNS",
            "FLNSC": "FLNSC",
            "FSDS": "FSDS",
            "FSDSC": "FSDSC",
            "FSNS": "FSNS",
            "FSNSC": "FSNSC",
            "LHFLX": "LHFLX",
            "SHFLX": "SHFLX",
            "QFLX": "QFLX",
            "TREFHT": "TREFHT",
            "TS": "TS",
            "PRECSL": "PRECSL",
            "PRECC": "PRECC",
            "PRECL": "PRECL",
        },
        "GPCP": {"PRECT": "PRECT"},
        "GPM":  {"PRECT": "PRECT"},
        "NOAA-OLR": {"FLUT": "FLUT"},
    }

    # config knobs
    ens_start  = 1
    ens_prefix = "EN"
    ens_width  = 2
    overwrite  = False
    verbose = True
    
    # region selection used for base computation
    regions = [dict(name="GLOBAL", lat=(-90, 90), lon=(0, 360))]

    # file naming template for model time series
    model_template = "%(var)s.%(ens)s.%(year)d.nc"

    # build exp metadata once
    exp_dict = build_experiments(data_path)

    groups = list(exp_list.keys())
    for group in [groups[0]]:
        period = exp_list[group]["period"]
        run    = exp_list[group]["run"]
        freq   = exp_list[group]["freq"]
        init_date = exp_list[group]["init_date"]
        windows = build_windows_from_start(init_date, lead_bins)
        print(group, "windows:")
        for k,(s,e) in windows.items():
            print("  ", k, s, e)
            
        # optional: separate dirs
        group_out = os.path.join(out_path, group)
        os.makedirs(group_out, exist_ok=True)
        base_cfg = WindowedTCCBaseConfig(
            windows=windows,
            regions=regions,
            out_dir=group_out,        
            overwrite=overwrite,
            verbose=verbose,
        )
        builder = S2SWindowedTCCBaseBuilder(base_cfg)
        
        # base builder (writes one file per exp,var,ens)
        for obs, model_to_obs_map in s2s_var_dict.items():
            
            # 1) collect files for this (group, obs)
            cfg = FileCollectionConfig(
                group=group,
                freq=freq,
                run=run,
                obs=obs,
                period=period,
                model_template=model_template,
                ens_prefix=ens_prefix,
                ens_width=ens_width,
                ens_start=ens_start,
            )

            collector = S2SFileCollector(
                exp_list=exp_list,
                exp_dict=exp_dict,
                obs_registry=OBS_REGISTRY,
                s2s_var_dict=model_to_obs_map,   # <-- mapping model_var -> obs_var
                get_obs_file_func=get_obs_file,
            )

            all_files = collector.collect(cfg, verbose=True)

            # 2) build base files per (exp,var,ens)
            for var in all_files.keys():
                for exp in all_files[var].keys():
                    # ---- per-member base files ----
                    for ens in all_files[var][exp]["model"].keys():
                        try:
                            builder.build_member_file(
                                all_files,
                                group=group,
                                freq=freq,
                                run=run,
                                obs_name=obs,
                                period=period,
                                var=var,
                                exp=exp,
                                ens=ens,
                                obs_var_override=model_to_obs_map,
                            )
                        except Exception as e:
                            print(f"[ERROR] member base failed: group={group} obs={obs} exp={exp} var={var} ens={ens} :: {e}")
            
                    # ---- ensemble-mean base file ----
                    try:
                        builder.build_ensmean_file(
                            all_files,
                            group=group,
                            freq=freq,
                            run=run,
                            obs_name=obs,
                            period=period,
                            var=var,
                            exp=exp,
                            obs_var_override=model_to_obs_map,
                        )
                    except Exception as e:
                        print(f"[ERROR] ensmean base failed: group={group} obs={obs} exp={exp} var={var} :: {e}")

Jan2012 windows:
   week1-2 2012-01-01 2012-01-14
   week3-4 2012-01-15 2012-01-28
   week5-6 2012-01-29 2012-02-11
   week7-8 2012-02-12 2012-02-25
[SKIP EXISTS] /compyfs/zhan391/v3_dart_cda_scratch/diag_output/tcc_rmse_base/Jan2012/s2s_base_tcc_rmse_Jan2012_daily_fc_ERA5_201201-201203_ALL_CTRL10-S0_EN01_UBOT.nc
[SKIP EXISTS] /compyfs/zhan391/v3_dart_cda_scratch/diag_output/tcc_rmse_base/Jan2012/s2s_base_tcc_rmse_Jan2012_daily_fc_ERA5_201201-201203_ALL_CTRL10-S0_EN02_UBOT.nc
[SKIP EXISTS] /compyfs/zhan391/v3_dart_cda_scratch/diag_output/tcc_rmse_base/Jan2012/s2s_base_tcc_rmse_Jan2012_daily_fc_ERA5_201201-201203_ALL_CTRL10-S0_EN03_UBOT.nc
[SKIP EXISTS] /compyfs/zhan391/v3_dart_cda_scratch/diag_output/tcc_rmse_base/Jan2012/s2s_base_tcc_rmse_Jan2012_daily_fc_ERA5_201201-201203_ALL_CTRL10-S0_EN04_UBOT.nc
[SKIP EXISTS] /compyfs/zhan391/v3_dart_cda_scratch/diag_output/tcc_rmse_base/Jan2012/s2s_base_tcc_rmse_Jan2012_daily_fc_ERA5_201201-201203_ALL_CTRL10-S0_EN05_UBOT.nc
[SKIP EXISTS] /compyf

/qfs/projects/ops/rh7/apps/E3SM/conda_envs/e3smu_1_12_0/compy/conda/envs/e3sm_unified_1.12.0_login/lib/python3.13/site-packages/dask/_task_spec.py:759: RuntimeWarning: invalid value encountered in divide
  return self.func(*new_argspec)


[SAVED] /compyfs/zhan391/v3_dart_cda_scratch/diag_output/tcc_rmse_base/Jan2012/s2s_base_tcc_rmse_Jan2012_daily_fc_ERA5_201201-201203_ALL_CTRL10-S0_EN01_PRECT.nc
[UNIT FIX] PRECT: m/s -> mm/day (×86400×1000)


/qfs/projects/ops/rh7/apps/E3SM/conda_envs/e3smu_1_12_0/compy/conda/envs/e3sm_unified_1.12.0_login/lib/python3.13/site-packages/dask/_task_spec.py:759: RuntimeWarning: invalid value encountered in divide
  return self.func(*new_argspec)


[SAVED] /compyfs/zhan391/v3_dart_cda_scratch/diag_output/tcc_rmse_base/Jan2012/s2s_base_tcc_rmse_Jan2012_daily_fc_ERA5_201201-201203_ALL_CTRL10-S0_EN02_PRECT.nc
[UNIT FIX] PRECT: m/s -> mm/day (×86400×1000)


...

[SAVED] /compyfs/zhan391/v3_dart_cda_scratch/diag_output/tcc_rmse_base/Jan2012/s2s_base_tcc_rmse_Jan2012_daily_fc_GPM_201201-201203_ALL_DART40-S0_EN39_PRECT.nc
[UNIT FIX] PRECT: m/s -> mm/day (×86400×1000)
[UNIT FIX] PRECT: m/s -> mm/day (×86400×1000)


/qfs/projects/ops/rh7/apps/E3SM/conda_envs/e3smu_1_12_0/compy/conda/envs/e3sm_unified_1.12.0_login/lib/python3.13/site-packages/dask/_task_spec.py:759: RuntimeWarning: invalid value encountered in divide
  return self.func(*new_argspec)


[SAVED] /compyfs/zhan391/v3_dart_cda_scratch/diag_output/tcc_rmse_base/Jan2012/s2s_base_tcc_rmse_Jan2012_daily_fc_GPM_201201-201203_ALL_DART40-S0_EN40_PRECT.nc
[UNIT FIX] PRECT: m/s -> mm/day (×86400×1000)
[UNIT FIX] PRECT: m/s -> mm/day (×86400×1000)
[UNIT FIX] PRECT: m/s -> mm/day (×86400×1000)
[UNIT FIX] PRECT: m/s -> mm/day (×86400×1000)
[UNIT FIX] PRECT: m/s -> mm/day (×86400×1000)
[UNIT FIX] PRECT: m/s -> mm/day (×86400×1000)
[UNIT FIX] PRECT: m/s -> mm/day (×86400×1000)
[UNIT FIX] PRECT: m/s -> mm/day (×86400×1000)
[UNIT FIX] PRECT: m/s -> mm/day (×86400×1000)
[UNIT FIX] PRECT: m/s -> mm/day (×86400×1000)
[UNIT FIX] PRECT: m/s -> mm/day (×86400×1000)
[UNIT FIX] PRECT: m/s -> mm/day (×86400×1000)
[UNIT FIX] PRECT: m/s -> mm/day (×86400×1000)
[UNIT FIX] PRECT: m/s -> mm/day (×86400×1000)
[UNIT FIX] PRECT: m/s -> mm/day (×86400×1000)
[UNIT FIX] PRECT: m/s -> mm/day (×86400×1000)
[UNIT FIX] PRECT: m/s -> mm/day (×86400×1000)
[UNIT FIX] PRECT: m/s -> mm/day (×86400×1000)
[UNIT FIX] P

/qfs/projects/ops/rh7/apps/E3SM/conda_envs/e3smu_1_12_0/compy/conda/envs/e3sm_unified_1.12.0_login/lib/python3.13/site-packages/dask/_task_spec.py:759: RuntimeWarning: invalid value encountered in divide
  return self.func(*new_argspec)


[SAVED] /compyfs/zhan391/v3_dart_cda_scratch/diag_output/tcc_rmse_base/Jan2012/s2s_base_tcc_rmse_Jan2012_daily_fc_GPM_201201-201203_ALL_DART40-S0_ENSMEAN_PRECT.nc
[SAVED] /compyfs/zhan391/v3_dart_cda_scratch/diag_output/tcc_rmse_base/Jan2012/s2s_base_tcc_rmse_Jan2012_daily_fc_NOAA-OLR_201201-201203_ALL_CTRL10-S0_EN01_FLUT.nc
[SAVED] /compyfs/zhan391/v3_dart_cda_scratch/diag_output/tcc_rmse_base/Jan2012/s2s_base_tcc_rmse_Jan2012_daily_fc_NOAA-OLR_201201-201203_ALL_CTRL10-S0_EN02_FLUT.nc
[SAVED] /compyfs/zhan391/v3_dart_cda_scratch/diag_output/tcc_rmse_base/Jan2012/s2s_base_tcc_rmse_Jan2012_daily_fc_NOAA-OLR_201201-201203_ALL_CTRL10-S0_EN03_FLUT.nc
[SAVED] /compyfs/zhan391/v3_dart_cda_scratch/diag_output/tcc_rmse_base/Jan2012/s2s_base_tcc_rmse_Jan2012_daily_fc_NOAA-OLR_201201-201203_ALL_CTRL10-S0_EN04_FLUT.nc
[SAVED] /compyfs/zhan391/v3_dart_cda_scratch/diag_output/tcc_rmse_base/Jan2012/s2s_base_tcc_rmse_Jan2012_daily_fc_NOAA-OLR_201201-201203_ALL_CTRL10-S0_EN05_FLUT.nc
[SAVED] /compyfs/